# Aula 08 · Jacobi e Gauss-Seidel

Esta aula apresenta o [capítulo 8 do site](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/). A ideia central: **sistemas gigantes, em que cada equação envolve só uns poucos vizinhos, se resolvem chutando e melhorando** — equação por equação, até parar de mudar.

**Ao fim da aula você consegue:**

1. deduzir os métodos de Jacobi e Gauss-Seidel e aplicá-los à mão;
2. escrever os dois métodos para qualquer sistema, com critério de parada;
3. verificar a dominância diagonal e reordenar as equações quando ela falha;
4. resolver problemas em grade (placa aquecida) sem montar a matriz.

**Roteiro:** 🧩 · 1. 🧑‍🏫 Jacobi · 2. 🧑‍🏫 Gauss-Seidel · 3. qualquer sistema · 4. dominância · 5. a placa · 6. outra área · 🎯 prática · 🧩 o cabo · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da aula

> **Telecomunicações — tensão na ponta de um cabo.**
>
> *Um sensor de nível num poço distante é alimentado por um cabo antigo, com a
> isolação ruim: em cada trecho, um pouco da corrente "vaza" para a terra. A fonte
> tem 10 V, e o sensor precisa de **pelo menos 1 V** para ligar. O técnico
> pergunta: "**com 30 trechos de cabo, chega tensão suficiente na ponta?**"*

Cada ponto do cabo dá uma equação que liga a tensão dele à dos dois vizinhos: um
sistema com 30 incógnitas, e só 3 delas em cada equação. No fim da aula, você o
resolve por Gauss-Seidel, sem montar matriz nenhuma.

## 1. No quadro: o método de Jacobi

O sistema de exemplo:
$3x_1 - 0{,}1x_2 - 0{,}2x_3 = 7{,}85$,
$0{,}1x_1 + 7x_2 - 0{,}3x_3 = -19{,}3$,
$0{,}3x_1 - 0{,}2x_2 + 10x_3 = 71{,}4$ (solução: $3$; $-2{,}5$; $7$).

📖 [capítulo 8 · No quadro: o método de Jacobi](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#no-quadro-o-metodo-de-jacobi)

### 🧑‍🏫 No quadro — o método de Jacobi

Caderno de papel aberto. No quadro:

1. isolar em cada equação $i$ a incógnita $x_i$;
2. chutar $x = (0, 0, 0)$;
3. calcular os valores novos com os **antigos**;
4. repetir até o maior $\varepsilon_a$ ficar abaixo da tolerância;
5. duas iterações à mão.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$ x_i = \frac{b_i - \sum_{j \neq i} A_{ij}\,x_j}{A_{ii}} $$

Iteração 1, de $(0, 0, 0)$: $x_1 = 7{,}85/3 = 2{,}6167$, $x_2 = -19{,}3/7 = -2{,}7571$,
$x_3 = 71{,}4/10 = 7{,}14$.

</details>

**✍️ Passo 1.** Com `x1, x2, x3 = 0.0, 0.0, 0.0`, calcule `novo1`, `novo2`, `novo3` (cada equação isolada, usando os valores **antigos**), depois faça `x1, x2, x3 = novo1, novo2, novo3` e imprima.

In [ ]:
# ✍️ passo 1

**Preveja:** o resultado bate com a iteração 1 do quadro?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Bate: `2.6167 -2.7571 7.14`. Jacobi usa só os valores antigos, que eram zero.

</details>

**✍️ Passo 2.** Ponha o passo 1 num laço de 10 iterações, imprimindo os três valores a cada volta.

In [ ]:
# ✍️ passo 2

**Preveja:** em quantas iterações a solução aparece com 6 casas?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Em **6**: `3.000000 -2.500000 7.000000`. Nenhuma matriz foi transformada: só três
contas repetidas.

📖 [capítulo 8 · No quadro: o método de Jacobi](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#no-quadro-o-metodo-de-jacobi)

</details>

> ⚠️ **Armadilha.** Escrever `x1 = ...` direto (em vez de `novo1 = ...`) e usar esse `x1` já na conta
de `x2` **não** é Jacobi: é Gauss-Seidel. Os dois convergem aqui, mas é bom saber
qual se está fazendo.

### 🎯 Sua vez — Uma iteração de Jacobi

Escreva `iteracao_jacobi(x1, x2, x3)`, que devolve a tupla `(novo1, novo2, novo3)` de uma iteração de Jacobi no sistema do exemplo.

In [ ]:
def iteracao_jacobi(x1, x2, x3):
    # sua solução aqui
    pass

In [ ]:
confere(iteracao_jacobi, [
    ((0.0, 0.0, 0.0), (2.6166666666666667, -2.757142857142857, 7.140000000000001)),
    ((3.0, -2.5, 7.0), (3.0, -2.5, 7.0)),
])

<details>
<summary><b>💡 Dica</b></summary>

As três contas do passo 1, com `return (novo1, novo2, novo3)`. Na solução exata, a iteração não muda nada.

</details>

## 2. Gauss-Seidel

📖 [capítulo 8 · Gauss-Seidel](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#gauss-seidel)

### 🧑‍🏫 No quadro — a melhoria de Gauss-Seidel

Caderno de papel aberto. No quadro:

1. mesma fórmula de Jacobi;
2. cada valor novo substitui o antigo **na hora**, já na próxima equação;
3. por que costuma convergir mais depressa.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

Em código: atualizar `x1` **antes** de calcular `x2`. Costuma precisar de cerca de
metade das iterações de Jacobi e gasta menos memória.

</details>

**✍️ Passo 3.** Repita o laço do passo 2, mas atualizando `x1`, `x2` e `x3` **direto**, uma equação depois da outra (sem `novo1`...).

In [ ]:
# ✍️ passo 3

**Preveja:** Gauss-Seidel chega à solução em mais ou menos iterações que Jacobi?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**Menos**: 4 contra 6. Já na primeira iteração, $x_2 = -2{,}79$ usa o $x_1$ novo, e
fica mais perto do $-2{,}5$.

📖 [capítulo 8 · Gauss-Seidel](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#gauss-seidel)

</details>

## 3. Para qualquer sistema

Escrito equação por equação, o método só serve para um sistema. Com a fórmula do
quadro num laço sobre as linhas da matriz, serve para qualquer um.

📖 [capítulo 8 · Para qualquer sistema](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#para-qualquer-sistema)

### 🎯 Sua vez — O valor novo de uma linha

Escreva `valor_novo(A, b, x, i)`, que devolve $\dfrac{b_i - \sum_{j \neq i} A_{ij}x_j}{A_{ii}}$ — a peça que Jacobi e Gauss-Seidel repetem.

In [ ]:
def valor_novo(A, b, x, i):
    # sua solução aqui
    pass

In [ ]:
A3 = np.array([[3.0, -0.1, -0.2], [0.1, 7.0, -0.3], [0.3, -0.2, 10.0]])
b3 = np.array([7.85, -19.3, 71.4])
confere(valor_novo, [
    ((A3, b3, np.array([1.0, 1.0, 1.0]), 1), -2.7285714285714286),
    ((A3, b3, np.array([3.0, -2.5, 7.0]), 2), 7.0),
])

<details>
<summary><b>💡 Dica</b></summary>

Comece com `soma = b[i]` e, num laço em `j`, subtraia `A[i, j] * x[j]` quando `j != i`.

</details>

**✍️ Passo 4.** Escreva um Gauss-Seidel genérico: `x = np.zeros(3)` e, em cada iteração, para cada linha `i`, faça `x[i] = valor_novo(A3, b3, x, i)`. Rode 10 iterações e imprima `x`.

In [ ]:
# ✍️ passo 4

**Preveja:** o resultado é o mesmo do passo 3?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

É: `[ 3.  -2.5  7. ]`. Como `x[i]` é atualizado na hora, é Gauss-Seidel. Para ser
Jacobi, seria preciso guardar os novos num array à parte.

📖 [capítulo 8 · Para qualquer sistema](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#para-qualquer-sistema)

</details>

## 4. Dominância diagonal

O sistema $x + 3y = 7$, $4x + y = 6$ tem solução $(1, 2)$.

📖 [capítulo 8 · Dominância diagonal](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#dominancia-diagonal)

**✍️ Passo 5.** Faça 5 iterações de Gauss-Seidel isolando `x = 7 - 3 * y` (da primeira equação) e `y = 6 - 4 * x` (da segunda), a partir de zero. Depois, troque: `x = (6 - y) / 4` e `y = (7 - x) / 3`.

In [ ]:
# ✍️ passo 5

**Preveja:** as duas ordens convergem?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A primeira **explode** (x passa de 100 000); a segunda converge para $(1, 2)$. A
diferença: na segunda, cada incógnita sai da equação em que tem o **maior**
coeficiente. Essa é a **dominância diagonal**: $|A_{ii}| > \sum_{j\neq i}|A_{ij}|$.

📖 [capítulo 8 · Dominância diagonal](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#dominancia-diagonal)

</details>

### 🎯 Sua vez — Tem dominância?

Escreva `diagonal_dominante(A)`, que devolve `True` se, em todas as linhas, `abs(A[i, i])` for **maior** que a soma dos `abs` dos outros elementos da linha.

In [ ]:
def diagonal_dominante(A):
    # sua solução aqui
    pass

In [ ]:
confere(diagonal_dominante, [
    ((A3,), True),
    ((np.array([[1.0, 3.0], [4.0, 1.0]]),), False),
    ((np.array([[4.0, 1.0], [1.0, 3.0]]),), True),
])

<details>
<summary><b>💡 Dica</b></summary>

Para cada linha, um acumulador dos `abs` fora da diagonal; se `abs(A[i, i]) <= soma`, `return False`.

</details>

## 5. Uma placa aquecida

Numa placa com as bordas a temperaturas fixas, em equilíbrio, cada ponto de dentro
tem a **média dos quatro vizinhos**. Gauss-Seidel direto na grade: percorrer os
pontos, trocando cada um pela média — sem montar matriz nenhuma.

📖 [capítulo 8 · Uma placa aquecida](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#uma-placa-aquecida)

> 🧰 **Comando novo: `np.zeros` com duas dimensões**
>
> `np.zeros((linhas, colunas))`, com um **par** de números, cria uma matriz de zeros
> pronta para ser preenchida com `T[i, j] = ...`.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
T = np.zeros((3, 4))
T[0, 1] = 5.0
print(T)

**✍️ Passo 6.** Crie `T = np.zeros((7, 7))`, ponha 100 em toda a linha 0 e faça 100 iterações: em cada uma, para `i` e `j` de 1 a 5, `T[i, j]` vira a média de `T[i-1, j]`, `T[i+1, j]`, `T[i, j-1]` e `T[i, j+1]`. Imprima `np.round(T, 1)`.

In [ ]:
# ✍️ passo 6

**Preveja:** quanto vale o ponto central, `T[3, 3]`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**25,0** — exatamente a média das quatro bordas (100, 0, 0, 0). A temperatura cai
da borda quente para a fria, mais depressa perto da quente.

📖 [capítulo 8 · Uma placa aquecida](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#uma-placa-aquecida)

</details>

## 6. Mesmo método, outra área

**A web.** No PageRank, a importância de cada página é a soma do que ela recebe de
quem aponta para ela — um sistema linear com uma equação por página, resolvido por
iteração porque são bilhões de páginas.

📖 [capítulo 8 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#mesmo-metodo-outra-area)

**✍️ Passo 7.** Quatro páginas, com links 0→1, 0→2, 1→2, 2→0 e 3→2, e `d = 0.85`. Comece com `r = [0.25, 0.25, 0.25, 0.25]` e itere 50 vezes (Jacobi): `r0 = 0.15/4 + d*r2`, `r1 = 0.15/4 + d*r0/2`, `r2 = 0.15/4 + d*(r0/2 + r1 + r3)`, `r3 = 0.15/4`.

In [ ]:
# ✍️ passo 7

**Preveja:** qual página fica mais importante? E a 0, que recebe um link só?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A página 2 (0,394), que recebe três links. A 0 vem logo atrás (0,373): o único
link que ela recebe vem da página mais importante, que só aponta para ela.

📖 [capítulo 8 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/08-jacobi-gauss-seidel/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

A prática desta aula é a própria resolução do problema, logo abaixo — um Gauss-Seidel completo, com critério de parada. Antes dela, confira a dominância do cabo: em cada ponto, o coeficiente da diagonal é $2 + R/R_{\text{fuga}} = 2{,}01$, e os vizinhos somam $2$. Tem dominância, mas **por pouco**. O que isso deve fazer com o número de iterações?

<details>
<summary><b>▶ Resposta</b></summary>

Deve aumentar muito: com a diagonal quase igual à soma dos vizinhos, cada iteração corrige muito pouco o erro. Guarde a previsão e confira no fim.

</details>

## 🧩 Resolvendo o problema

> *"**Com 30 trechos de cabo, chega tensão suficiente na ponta?**"* — o técnico.

Em cada ponto $i$ do meio, a corrente que chega do vizinho de trás, mais a que chega
do da frente, é igual à que vaza para a terra:

$$ \frac{V_{i-1} - V_i}{R} + \frac{V_{i+1} - V_i}{R} = \frac{V_i}{R_{\text{fuga}}}
\quad\Longrightarrow\quad V_i = \frac{V_{i-1} + V_{i+1}}{2 + R/R_{\text{fuga}}}. $$

Na ponta (ponto $N$), só há o vizinho de trás: $V_N = \dfrac{V_{N-1}}{1 + R/R_{\text{fuga}}}$.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Um cabo longo, modelado como 30 trechos: cada trecho tem 1 ohm de resistência
# em série, e em cada ponto há um "vazamento" de 100 ohm para a terra (umidade,
# isolamento velho). A fonte, no ponto 0, tem 10 V; a ponta (ponto 30) está aberta.
N = 30
R_SERIE = 1.0
R_FUGA = 100.0
V_FONTE = 10.0
V_MINIMA = 1.0       # o sensor da ponta precisa de pelo menos 1 V para ligar

### 🎯 Sua vez — A tensão na ponta

Escreva `tensao_na_ponta(tol)`, que cria `V = np.zeros(N + 1)` com `V[0] = V_FONTE`,
aplica Gauss-Seidel nos pontos 1 a $N-1$ (fórmula do meio) e no ponto $N$ (fórmula
da ponta), e para quando a maior mudança de uma iteração for menor que `tol` (no
máximo 20 000 iterações). Devolva a tupla `(V[N], número de iterações)`.

In [ ]:
def tensao_na_ponta(tol):
    # sua solução aqui
    pass

In [ ]:
confere(tensao_na_ponta, [
    ((1e-9,), (0.9474358926416561, 1320)),
])

<details>
<summary><b>💡 Dica</b></summary>

É a placa aquecida em uma dimensão: um laço de iterações, dentro dele um laço de
`i = 1` a `N - 1`, e depois a conta do ponto `N`. Guarde a maior mudança
(padrão extremo) para o critério de parada.

</details>

A resposta para o técnico:

In [ ]:
resposta = tensao_na_ponta(1e-9)
if resposta is not None:
    v_ponta, iteracoes = resposta
    print("tensão na ponta:", v_ponta, "V   (mínimo:", V_MINIMA, "V)")
    print("iterações:", iteracoes)

<details>
<summary><b>▶ O que os números dizem</b></summary>

Chegam só **0.947 V** na ponta — abaixo do 1 V de que o sensor precisa. Mais
de 90 % da tensão se perde no caminho, entre a resistência do cabo e os vazamentos.
O técnico precisa de uma fonte maior, de um cabo novo ou de um cabo mais curto.

E o número de iterações (1320) confirma a previsão da prática: com a dominância
diagonal **por um fio** (2,01 contra 2), cada iteração corrige muito pouco. Um cabo
com isolação melhor (vazamento menor) teria dominância ainda mais fraca e convergência
ainda mais lenta. Para problemas assim, existem variantes aceleradas do Gauss-Seidel
(como o SOR, a sobrerrelaxação).

</details>

## 📋 A lista

Abra a [Lista 08](https://lacouth.github.io/metodos_telecom-site/listas/lista08/). O **Exercício 01** é à mão (✏️): duas iterações de Jacobi e
duas de Gauss-Seidel. Comece por ele, no papel.

**a)** O sistema $10x + y = 12$, $x + 5y = -4$ tem dominância diagonal?

<details>
<summary><b>▶ Resposta</b></summary>

Tem: $|10| > |1|$ e $|5| > |1|$.

</details>

Termine o exercício e siga para o **Exercício 02**, a dominância como função.

## 🚪 Antes de sair

**1.** Qual a diferença, em uma frase, entre Jacobi e Gauss-Seidel?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Jacobi usa só os valores da iteração anterior; Gauss-Seidel usa cada valor novo assim que ele é calculado.

</details>

**2.** Por que métodos iterativos são preferidos para sistemas enormes, como o da placa com um milhão de pontos?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Porque cada equação envolve só alguns vizinhos: uma iteração custa pouco, e não é preciso guardar nem transformar a matriz inteira, como na eliminação de Gauss.

</details>

**3.** O que fazer se o sistema não tem dominância diagonal?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Tentar reordenar as equações para que cada incógnita saia da equação em que tem o maior coeficiente. Se não der, usar eliminação de Gauss — ou iterar sem garantia, conferindo se converge.

</details>

## 🏠 Para casa

- Refaça no papel duas iterações de Jacobi e de Gauss-Seidel **sem olhar**.
- Termine a [Lista 08](https://lacouth.github.io/metodos_telecom-site/listas/lista08/).
- A próxima aula fecha a unidade com a parte que nenhuma biblioteca faz: **montar** o
  sistema a partir de um problema escrito.